# Корпус PyTorch docs для RAG

1. Настройки и список файлов
2. Функции загрузки и чанкинга
3. Сборка корпуса → `chunks.json`
4. Имена API для docstring'ов
5. Сбор docstring'ов

In [22]:
import re, json, random, time, textwrap, importlib, inspect
from pathlib import Path
from collections import Counter, defaultdict

DOCS = Path.home() / 'pytorch' / 'docs' / 'source'  # docs/source из клона github.com/pytorch/pytorch
files = list(DOCS.rglob("*.md"))
print(len(files))

256


## Функции разбора

In [23]:
import textwrap

def split_directives(block):
    result = []
    header = None
    body = []
    for line in block.splitlines():
        if line.startswith('.. '):
            if header:
                result.append((header, textwrap.dedent("\n".join(body)).strip()))

            header = line
            body = []
        else:
            body.append(line)
    if header:
        result.append((header, textwrap.dedent("\n".join(body)).strip()))
    return result

In [24]:
def parse_header(header):
    m = re.match(r"\.\. ([\w:-]+)::(?:\s+([\w.]+))?", header)
    if m:
        return (m.group(1), m.group(2))
    else:
        return None

In [25]:
POINTERS = {"autofunction", "autoclass", "automethod", "autoattribute",
            "autodata", "autoexception"}

SERVICE = {"toctree", "role", "image", "raw", "contents"}

CONTENT_KINDS = {"note", "warning", "seealso", "deprecated", "hint", "math"}

def full_name(module, name):
    if module is None:
        return name
    if not name.startswith("torch."):
        name =  module + "." + name
    return name

def process_file(path):
    text = path.read_text(encoding="utf-8")
    current_module = None
    chunks, pointers = [], []
    for block in re.findall(r"`{3,}\{eval-rst\}(.*?)`{3,}", text, flags=re.DOTALL):
        for header, body in split_directives(block):
            parsed = parse_header(header)
            if parsed is None:
                continue
            kind, name = parsed
            if kind in ("currentmodule", "automodule", "py:module"):
                current_module = name
            elif kind in POINTERS:
                pointers.append(full_name(current_module, name))
            elif kind == "autosummary":
                for line in body.splitlines():
                    line = line.strip()
                    if line.strip() and not line.strip().startswith(":"):
                        pointers.append(full_name(current_module, line))
            elif kind in SERVICE:
                continue
            else:
                if kind in CONTENT_KINDS:
                    rest = header.split("::", 1)[1].strip()
                    text_ = (rest + "\n" + body).strip()
                    chunks.append({"module": current_module, "name": None, "kind": kind, "text": text_})
                else:
                    if name:
                        pointers.append(full_name(current_module, name))
                    chunks.append({"module": current_module, "name": name, "kind": kind, "text": body})
    return chunks, pointers

In [26]:
def split_sections(text):
    sections = []
    stack, lines, fence = [], [], 0
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"^(`{3,})", s)
        if m:
            if fence == 0:
                fence = len(m.group(1))
            elif re.fullmatch(r"`{3,}", s) and len(s) >= fence:
                fence = 0
            lines.append(line)
            continue
        if fence == 0:
            h = re.match(r"^(#{1,6})\s+(.*)$", line)
            if h:
                level = len(h.group(1))
                title = h.group(2)
                body = "\n".join(lines).strip()
                if body:
                    sections.append({"path": list(stack), "text": body})
                stack = stack[:level-1] + [title]
                lines = []
                continue
        lines.append(line)
    body = "\n".join(lines).strip()
    if body:
        sections.append({"path": list(stack), "text": body})
    return sections

In [27]:
def replace_open(m):
    kind, arg = m.group(1), m.group(2)
    if kind in ("note", "hint", "warning"):
        return kind.capitalize()+":"
    if kind == "tab-item":
        return "Tab: "+arg
    return ""

def clean_colon_blocks(text):
    text = re.sub(r"^:{3,}\{([\w-]+)\}[ \t]*(.*)$", replace_open, text, flags=re.M)
    text = re.sub(r"^:{3,}[ \t]*$", "", text, flags=re.M)
    return text

def clean_misc(text):
    text = re.sub(r"^%.*$", "", text, flags=re.M)                      
    text = re.sub(r"^\([^)\n]+\)=[ \t]*$", "", text, flags=re.M)       
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    text = re.sub(r"^`{3,}\{role\}.*?^`{3,}[ \t]*$", "", text, flags=re.M | re.DOTALL)
    text = re.sub(r"^[ \t]*:class:.*$", "", text, flags=re.M)
    return text

def clean_roles(text):
    text = re.sub(r"\{[\w:-]+\}(`[^`]+`)", r"\1", text)
    text = re.sub(r"`~", "`", text)
    return text

EVAL_RST = r"`{3,}\{eval-rst\}.*?`{3,}"

def load_file(path):
    text = path.read_text(encoding="utf-8")
    api_chunks, pointers = process_file(path)
    clean = re.sub(EVAL_RST, "", text, flags=re.DOTALL)
    clean = clean_colon_blocks(clean)
    clean = clean_misc(clean)
    clean = clean_roles(clean)
    clean = clean_rst(clean)
    sections = split_sections(clean)
    return sections, api_chunks, pointers

In [28]:
def split_units(text):
    units, cur, fence = [], [], 0
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"^(`{3,})", s)
        if m:
            if fence == 0:
                fence = len(m.group(1))
            elif re.fullmatch(r"`{3,}", s) and len(s) >= fence:
                fence = 0
            cur.append(line)
            continue
        if fence == 0 and not s:
            body = "\n".join(cur).strip()
            if body:
                units.append(body)
            cur = []
            continue
        if fence == 0 and re.match(r"^([*-]|\d+[.)])\s", line) and cur:
            body = "\n".join(cur).strip()
            if body:
                units.append(body)
            cur = []
        cur.append(line)
    body = "\n".join(cur).strip()
    if body:
        units.append(body)
    return units

In [29]:
def split_long(text, max_chars=1500):
    chunks, cur = [], ""
    for unit in split_units(text):
        if not cur:
            cur = unit
        elif len(cur) + 2 + len(unit) <= max_chars:
            cur = cur + "\n\n" + unit
        else:
            chunks.append(cur)
            cur = unit
    if cur:
        chunks.append(cur)
    return chunks

In [30]:
def replace_directive(m):
    return m.group(1).capitalize() + ":"


def clean_rst(text):
    text = re.sub(r":[\w:-]+:(`[^`]+`)", r"\1", text)          # :class:`X` -> `X`
    text = re.sub(r"`~", "`", text)                             # `~a.b` -> `a.b`
    text = re.sub(r"(?<!`)``([^`\n]+)``(?!`)", r"`\1`", text)   # ``x`` -> `x` (не трогая ``` блоков кода)
    text = re.sub(r"\s*<[^<>`\s]+>`", "`", text)                # `текст<метка>` -> `текст`
    text = re.sub(r"^\.\. (note|warning|seealso|deprecated)::",
                  replace_directive, text, flags=re.M)          # .. note:: -> Note:
    return text

def build_chunks(files):
    result, seen = [], set()
    for file in files:
        sections, api_chunks, pointers = load_file(file)
        rel = str(file.relative_to(DOCS))
        for s in sections:
            title = " > ".join(s["path"])
            for p in split_long(s["text"]):
                if len(p.strip()) < 20:
                    continue
                text = title + "\n" + p
                if text in seen:
                    continue
                seen.add(text)
                result.append({"id":len(result), "file":rel, "kind": "text", "title": title, "text": text})
        for c in api_chunks:
            label = c["name"] or c["kind"]
            if c["module"]:
                title = c["module"] + "." + label
            else:
                title = label
            body = clean_rst(c["text"])
            if len(body.strip()) < 20:
                continue
            if c["kind"] in ("note", "warning", "seealso"):
                body = c["kind"].capitalize() + ":\n" + body
            text = title + "\n" + body
            if text in seen:
                continue
            seen.add(text)
            result.append({"id":len(result), "file":rel, "kind": "api", "title": title, "text": text})
    return result

## Сборка корпуса

In [31]:
chunks_all = build_chunks(files)
print(len(chunks_all), Counter(c["kind"] for c in chunks_all))
with open("../data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_all, f, ensure_ascii=False, indent=1)

2349 Counter({'text': 2223, 'api': 126})


## Имена API для docstring'ов

In [32]:
all_pointers = set()
for file in files:
    all_pointers.update(load_file(file)[2])

# убираем тильду из rst-имён: torch.nn.~parameter.Parameter -> torch.nn.parameter.Parameter
all_pointers = {n.replace("~", "") for n in all_pointers}

valid = re.compile(r"^torch(\.\w+)*$")
names = sorted(n for n in all_pointers if valid.match(n))
print(len(all_pointers), len(names))

4314 4309


## Docstring'и из пакета torch

In [33]:
import torch



def get_doc(full):
    parts = full.split(".")
    for k in range(len(parts), 0, -1):
        try:
            obj = importlib.import_module(".".join(parts[:k]))
        except Exception:
            continue
        try:
            for attr in parts[k:]:
                obj = getattr(obj, attr)
            return inspect.getdoc(obj)
        except Exception:
            return None
    return None

In [34]:
docs = {}
failed = []
t0 = time.time()
for i, n in enumerate(names):
    d = get_doc(n)
    if d:
        docs[n] = d
    else:
        failed.append(n)

with open("../data/docs.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=1)
print(len(docs), len(failed), round(time.time() - t0, 1))

3839 470 0.1


~/anaconda3/lib/python3.14/inspect.py:778: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  doc = object.__doc__


## Docstring'и → чанки и итоговый корпус

In [35]:
def docs_to_chunks(docs, existing, start_id):
    """docstring'и из пакета -> чанки в том же формате, что и остальной корпус."""
    seen_text = {c["text"] for c in existing}
    seen_body = set()
    have = {c["title"] for c in existing if c["kind"] == "api"}   # уже описано вручную в .md
    result = []
    for name, doc in docs.items():
        if name in have:
            continue
        for p in split_long(clean_rst(doc)):
            if len(p.strip()) < 20 or p in seen_body:     # пустое и алиасы с одинаковым текстом
                continue
            text = name + "\n" + p
            if text in seen_text:
                continue
            seen_text.add(text)
            seen_body.add(p)
            result.append({"id": start_id + len(result), "file": "docstring",
                           "kind": "api", "title": name, "text": text})
    return result


In [36]:
new_chunks = docs_to_chunks(docs, chunks_all, len(chunks_all))
chunks_full = chunks_all + new_chunks
print(len(chunks_all), len(new_chunks), len(chunks_full))
print(new_chunks[0]["text"][:400])

with open("../data/chunks_full.json", "w", encoding="utf-8") as f:
    json.dump(chunks_full, f, ensure_ascii=False, indent=1)

2349 4507 6856
torch.Event
Event(device=None, *, enable_timing=False, blocking=False, interprocess=False)

Query and record Stream status to identify or control dependencies across Stream and measure timing.

Arguments:
    device (`torch.device`, optional): the desired device for the Event.
        If not given, the current `accelerator` type will be used.
    enable_timing (bool, optional): indicates if the ev
